In [ ]:
# ============================================================
# PRUEBA DE HIPÓTESIS — INGENIERÍA DE SISTEMAS
# Script completo para Google Colab / Jupyter Notebook
# Autor: Análisis Estadístico Aplicado
# ============================================================
# INSTRUCCIONES COLAB:
# 1. Subir los archivos .xlsx al entorno de Colab
# 2. Ajustar las rutas en la sección "Cargar Datos"
# 3. Ejecutar todas las celdas en orden
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paleta de colores profesional ──────────────────────────
C1 = '#2E86AB'   # Azul
C2 = '#E84855'   # Rojo
C3 = '#3BB273'   # Verde
C4 = '#F4A261'   # Naranja
C5 = '#7B2D8B'   # Morado

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F5F5F5',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 1.2
})

# ============================================================
# CARGA DE DATOS
# ── Ajustar rutas según entorno ──────────────────────────
# ============================================================
# En Google Colab: subir archivos y usar solo el nombre
# Ejemplo:  pd.read_excel('tstudent_algoritmos.xlsx')

from google.colab import files
import io

print("📂 Sube el archivo: mannwhitney_redes.xlsx")
uploaded = files.upload()
df2 = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))

# ============================================================
# ╔══════════════════════════════════════════════════════╗
# ║  HIPÓTESIS 2 — Mann-Whitney (Cola Inferior)          ║
# ║  H0: X ≥ Y   |   H1: X < Y                          ║
# ║  Dataset: mannwhitney_redes.xlsx                     ║
# ╚══════════════════════════════════════════════════════╝
# ============================================================

X = df2['Red_X_ms']
Y = df2['Red_Y_ms']

print("\n" + "=" * 65)
print("HIPÓTESIS 2: Mann-Whitney U (Cola Inferior)")
print("=" * 65)

print("\n📊 ESTADÍSTICAS DESCRIPTIVAS")
print(f"{'':25s} {'Red X (ms)':>15s} {'Red Y (ms)':>15s}")
print(f"{'Media':25s} {X.mean():>15.4f} {Y.mean():>15.4f}")
print(f"{'Mediana':25s} {X.median():>15.4f} {Y.median():>15.4f}")
print(f"{'Desviación estándar':25s} {X.std():>15.4f} {Y.std():>15.4f}")
print(f"{'Mínimo':25s} {X.min():>15.4f} {Y.min():>15.4f}")
print(f"{'Máximo':25s} {X.max():>15.4f} {Y.max():>15.4f}")
print(f"{'n':25s} {len(X):>15d} {len(Y):>15d}")

print("\n✅ VERIFICACIÓN DE SUPUESTOS")
sw_X = stats.shapiro(X)
sw_Y = stats.shapiro(Y)
print(f"Shapiro-Wilk X:  W = {sw_X.statistic:.4f}, p = {sw_X.pvalue:.4f} "
      f"→ {'Normal ✓' if sw_X.pvalue > 0.05 else 'No normal ✗'}")
print(f"Shapiro-Wilk Y:  W = {sw_Y.statistic:.4f}, p = {sw_Y.pvalue:.4f} "
      f"→ {'Normal ✓' if sw_Y.pvalue > 0.05 else 'No normal ✗'}")
print("  → Ambas distribuciones son normales, pero Mann-Whitney es")
print("    preferible por su robustez ante distribuciones asimétricas")
print("    y valores atípicos en datos de latencia de red.")

print("\n🔬 APLICACIÓN DE PRUEBA")
u_stat, p_h2 = stats.mannwhitneyu(X, Y, alternative='less')
print(f"U estadístico:         {u_stat:.2f}")
print(f"Valor p (cola inf.):   {p_h2:.8f}")
print(f"Nivel de sign. (α):    0.05")
print(f"\n📌 DECISIÓN: {'✅ RECHAZAR H0 — Latencia X < Latencia Y confirmado' if p_h2 < 0.05 else '❌ NO rechazar H0'}")
print(f"\n💡 INTERPRETACIÓN:")
print(f"   La Red X (mediana={X.median():.2f} ms) presenta latencias")
print(f"   significativamente menores que la Red Y (mediana={Y.median():.2f} ms).")
print(f"   Con p < 0.001, se rechaza H0 con alta significancia.")

fig2, axes = plt.subplots(1, 3, figsize=(16, 5))
fig2.suptitle('H2: Mann-Whitney — Latencia de Red (ms)',
              fontsize=13, fontweight='bold')

axes[0].hist(X, bins=20, color=C3, alpha=0.8, edgecolor='white')
axes[0].axvline(X.median(), color='darkgreen', lw=2, ls='--', label=f'Mediana={X.median():.1f}')
axes[0].set_title('Histograma — Red X'); axes[0].set_xlabel('ms')
axes[0].set_ylabel('Frecuencia'); axes[0].legend()

axes[1].hist(Y, bins=20, color=C4, alpha=0.8, edgecolor='white')
axes[1].axvline(Y.median(), color='darkorange', lw=2, ls='--', label=f'Mediana={Y.median():.1f}')
axes[1].set_title('Histograma — Red Y'); axes[1].set_xlabel('ms')
axes[1].set_ylabel('Frecuencia'); axes[1].legend()

bp2 = axes[2].boxplot([X, Y], labels=['Red X', 'Red Y'],
                      patch_artist=True, notch=True,
                      medianprops=dict(color='black', lw=2))
for box, color in zip(bp2['boxes'], [C3, C4]):
    box.set_facecolor(color); box.set_alpha(0.75)
axes[2].set_title('Boxplot Comparativo'); axes[2].set_ylabel('Latencia (ms)')

plt.tight_layout()
plt.savefig('H2_mannwhitney.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n" + "=" * 65)
print("RESUMEN EJECUTIVO — HIPÓTESIS 2")
print("=" * 65)

print(f"{'Hipótesis':<12s} {'Prueba':<20s} {'Estadístico':<18s} {'p-valor':<14s} {'Decisión'}")
print("-" * 65)

print(f"{'H2':<12s} {'Mann-Whitney':<20s} {f'U={u_stat:.2f}':<18s} {p_h2:<14.6f} "
      f"{'Rechazar H0 ✅' if p_h2 < 0.05 else 'No rechazar H0 ❌'}")

print("=" * 65)
print(f"α = 0.05  |  n = {len(X)}")